# 05. ETA / Total Trip Time Prediction

## Objective

Predict the total trip duration in minutes using only information available at or before trip start.

The model deliberately excludes `dropoff_timestamp` and any variables derived from the actual trip duration. This prevents target leakage and keeps the prediction setup aligned with the challenge requirement.

### Data-quality treatment

For the ETA task:

- Negative-duration trips are excluded because the target is invalid.
- Zero-duration trips are excluded because they cannot represent a meaningful travel time.
- Trips with calculated speed above 100 mph are excluded as unrealistic observations.
- Zero-distance trips are retained because distance can legitimately be recorded as zero or unreliable in some records.
- Raw records are not deleted from the underlying dataset. These filters apply only to the ETA modelling sample.

### Target

`trip_duration_minutes`

### Model inputs

The model uses trip and pickup-time information such as:

- provider
- rider count
- distance
- rate class
- offline-record indicator
- origin and destination zone IDs
- pickup hour
- day of week
- day of month
- month
- weekend indicator

`dropoff_timestamp`, financial settlement fields, calculated speed, and the target itself are excluded from the predictors.

In [3]:
# Core libraries for data loading, feature preparation, modelling and evaluation.
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import joblib

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")

# Project root.
BASE_DIR = Path(r"C:\Users\arudk\Downloads\UrbanFlow_AI")

# Raw taxi files.
TAXI_DIR = BASE_DIR / "data" / "raw" / "taxi"

# Model and output directories.
MODEL_DIR = BASE_DIR / "models" / "eta"
OUTPUT_DIR = BASE_DIR / "outputs"

MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", BASE_DIR)
print("Taxi data directory:", TAXI_DIR)
print("ETA model directory:", MODEL_DIR)

Project directory: C:\Users\arudk\Downloads\UrbanFlow_AI
Taxi data directory: C:\Users\arudk\Downloads\UrbanFlow_AI\data\raw\taxi
ETA model directory: C:\Users\arudk\Downloads\UrbanFlow_AI\models\eta


## 1. Build a Representative Modelling Sample

The complete taxi dataset contains more than 48 million records, so model development uses a representative sample from every monthly file.

A fixed sampling fraction is applied independently to each file. This preserves coverage across the full April 2025 to March 2026 study period rather than concentrating the training data in only a subset of months.

In [8]:
# Columns required for ETA modelling.
# dropoff_timestamp is loaded only to construct the target and quality checks.
# It will never be included as a model feature.
eta_columns = [
    "provider_code",
    "pickup_timestamp",
    "dropoff_timestamp",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
]

SAMPLE_FRAC = 0.02
CHUNKSIZE = 500_000
RANDOM_STATE = 42

taxi_files = sorted(TAXI_DIR.glob("*.csv"))

print(f"Monthly files found: {len(taxi_files)}")

eta_samples = []

for file_path in taxi_files:
    monthly_parts = []

    for chunk in pd.read_csv(
        file_path,
        usecols=eta_columns,
        chunksize=CHUNKSIZE,
        low_memory=False
    ):
        # Sample within each chunk so the large CSVs never need to
        # be loaded completely into memory.
        sampled = chunk.sample(
            frac=SAMPLE_FRAC,
            random_state=RANDOM_STATE
        )
        monthly_parts.append(sampled)

    monthly_sample = pd.concat(monthly_parts, ignore_index=True)
    eta_samples.append(monthly_sample)

    print(f"{file_path.name}: {len(monthly_sample):,} sampled rows")

eta_df = pd.concat(eta_samples, ignore_index=True)

print("\nFinal ETA sample:")
print("Rows:", f"{len(eta_df):,}")
print("Columns:", len(eta_df.columns))

Monthly files found: 12
Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv: 79,411 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv: 91,837 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv: 86,459 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv: 77,979 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv: 71,482 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv: 85,020 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv: 88,574 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv: 83,629 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv: 86,100 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2026-01.csv: 74,498 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2026-02.csv: 67,997 sampled rows
Urban_Flow_Analytics_Taxi_Dataset_2026-03.csv: 79,049 sampled rows

Final ETA sample:
Rows: 972,035
Columns: 9


## 2. Construct the Trip-Time Target and Apply Quality Filters

The prediction target is the elapsed time between pickup and drop-off:

`trip_duration_minutes = dropoff_timestamp - pickup_timestamp`

Before training, observations with invalid or clearly unreliable targets are removed from the modelling sample.

The filtering rules follow the data-quality assessment:

- Negative duration: invalid target, excluded.
- Zero duration: no meaningful travel time, excluded.
- Speed above 100 mph: treated as an unrealistic observation and excluded.
- Zero-distance trips with positive duration: retained because the distance field itself may be unreliable for these records.

The speed threshold is used only for quality control. Calculated speed is **not** passed to the model because it is derived from the target duration and would cause target leakage.

In [11]:
# Parse timestamps once so duration and calendar features use the same
# datetime representation throughout the ETA workflow.
eta_df["pickup_timestamp"] = pd.to_datetime(
    eta_df["pickup_timestamp"],
    errors="coerce"
)

eta_df["dropoff_timestamp"] = pd.to_datetime(
    eta_df["dropoff_timestamp"],
    errors="coerce"
)

# Calculate the prediction target in minutes.
eta_df["trip_duration_seconds"] = (
    eta_df["dropoff_timestamp"] - eta_df["pickup_timestamp"]
).dt.total_seconds()

eta_df["trip_duration_minutes"] = (
    eta_df["trip_duration_seconds"] / 60.0
)

# Calculate speed only for records where both duration and distance
# are positive. This is a quality-control variable, not a predictor.
eta_df["speed_mph"] = np.nan

valid_speed = (
    (eta_df["trip_duration_seconds"] > 0)
    & (eta_df["distance_miles"] > 0)
)

eta_df.loc[valid_speed, "speed_mph"] = (
    eta_df.loc[valid_speed, "distance_miles"]
    / (eta_df.loc[valid_speed, "trip_duration_seconds"] / 3600)
)

# Quality flags used to quantify the modelling exclusions.
eta_df["negative_duration_flag"] = (
    eta_df["trip_duration_seconds"] < 0
)

eta_df["zero_duration_flag"] = (
    eta_df["trip_duration_seconds"] == 0
)

eta_df["unrealistic_speed_flag"] = (
    eta_df["speed_mph"] > 100
)

# Keep only records with a valid positive duration and no unrealistic speed.
eta_model_df = eta_df[
    (eta_df["trip_duration_seconds"] > 0)
    & (~eta_df["unrealistic_speed_flag"])
].copy()

print("Original ETA sample:", f"{len(eta_df):,}")
print("Negative duration:", f"{eta_df['negative_duration_flag'].sum():,}")
print("Zero duration:", f"{eta_df['zero_duration_flag'].sum():,}")
print("Speed >100 mph:", f"{eta_df['unrealistic_speed_flag'].sum():,}")
print("Remaining modelling rows:", f"{len(eta_model_df):,}")

print("\nTrip-duration summary after filtering:")
display(
    eta_model_df["trip_duration_minutes"].describe(
        percentiles=[0.01, 0.25, 0.50, 0.75, 0.95, 0.99]
    ).to_frame("minutes")
)

Original ETA sample: 972,035
Negative duration: 53
Zero duration: 12,904
Speed >100 mph: 245
Remaining modelling rows: 958,833

Trip-duration summary after filtering:


,minutes
count,958833.000000
mean,18.027490
std,25.213485
min,0.016667
1%,0.716667
25%,8.483333
50%,14.016667
75%,22.266667
95%,45.050000
99%,73.266667


## 3. Feature Engineering

The ETA model must use information available before or at trip start.

Calendar features are extracted from `pickup_timestamp` to capture recurring demand and travel-time patterns. Categorical and numeric fields are converted into model-compatible numeric representations.

Importantly, variables derived from the completed trip are not used as predictors.

### Leakage controls

The following are deliberately excluded:

- `dropoff_timestamp`
- `trip_duration_seconds`
- `trip_duration_minutes`
- `speed_mph`
- `negative_duration_flag`
- `zero_duration_flag`
- `unrealistic_speed_flag`
- Any post-trip financial information

The calculated speed is especially important to exclude because:

`speed = distance / trip duration`

Therefore, providing speed to the model would reveal information about the target itself.

In [14]:
# Work on the filtered modelling dataset only.
eta_model_df["pickup_hour"] = eta_model_df["pickup_timestamp"].dt.hour
eta_model_df["pickup_day_of_week"] = eta_model_df["pickup_timestamp"].dt.dayofweek
eta_model_df["pickup_day_of_month"] = eta_model_df["pickup_timestamp"].dt.day
eta_model_df["pickup_month"] = eta_model_df["pickup_timestamp"].dt.month
eta_model_df["is_weekend"] = (
    eta_model_df["pickup_day_of_week"] >= 5
).astype(int)

# Convert fields that should be numeric.
numeric_columns = [
    "provider_code",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "origin_loc_id",
    "dest_loc_id",
]

for column in numeric_columns:
    eta_model_df[column] = pd.to_numeric(
        eta_model_df[column],
        errors="coerce"
    )

# Missing numeric values are filled using training-independent medians
# from the modelling sample. This keeps the feature matrix complete
# without inventing extreme values.
for column in numeric_columns:
    eta_model_df[column] = eta_model_df[column].fillna(
        eta_model_df[column].median()
    )

# Normalize the offline-record indicator.
offline_normalized = (
    eta_model_df["offline_record_flag"]
    .astype(str)
    .str.strip()
    .str.upper()
)

offline_mapping = {
    "Y": 1,
    "YES": 1,
    "TRUE": 1,
    "1": 1,
    "N": 0,
    "NO": 0,
    "FALSE": 0,
    "0": 0,
}

eta_model_df["offline_record_flag"] = (
    offline_normalized.map(offline_mapping)
    .fillna(-1)
    .astype(int)
)

# Predictor set.
eta_features = [
    "provider_code",
    "rider_count",
    "distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_day_of_month",
    "pickup_month",
    "is_weekend",
]

X = eta_model_df[eta_features].copy()
y = eta_model_df["trip_duration_minutes"].copy()

print("Feature matrix:", X.shape)
print("Target:", y.shape)

print("\nFeatures used:")
for feature in eta_features:
    print(" -", feature)

print("\nMissing feature values:")
print(X.isna().sum().sum())

print("\nTarget summary:")
display(y.describe().to_frame("trip_duration_minutes"))

Feature matrix: (958833, 12)
Target: (958833,)

Features used:
 - provider_code
 - rider_count
 - distance_miles
 - rate_class_id
 - offline_record_flag
 - origin_loc_id
 - dest_loc_id
 - pickup_hour
 - pickup_day_of_week
 - pickup_day_of_month
 - pickup_month
 - is_weekend

Missing feature values:
0

Target summary:


,trip_duration_minutes
count,958833.000000
mean,18.027490
std,25.213485
min,0.016667
25%,8.483333
50%,14.016667
75%,22.266667
max,1816.583333


## 4. Chronological Train, Validation and Test Split

The ETA model is evaluated using a time-ordered split rather than random sampling.

This reflects the real deployment scenario: the model is trained on historical trips and then used to predict trips occurring later.

The dataset is divided into:

- 70% training data
- 15% validation data
- 15% held-out test data

The validation set is used for model comparison and tuning. The test set remains untouched until the final model has been selected.

In [17]:
# Sort by pickup time before splitting so that future observations
# never enter the training set.
eta_model_df = eta_model_df.sort_values(
    "pickup_timestamp"
).reset_index(drop=True)

# Rebuild X and y after sorting.
X = eta_model_df[eta_features].copy()
y = eta_model_df["trip_duration_minutes"].copy()

n = len(eta_model_df)

train_end = int(n * 0.70)
validation_end = int(n * 0.85)

X_train = X.iloc[:train_end].copy()
y_train = y.iloc[:train_end].copy()

X_val = X.iloc[train_end:validation_end].copy()
y_val = y.iloc[train_end:validation_end].copy()

X_test = X.iloc[validation_end:].copy()
y_test = y.iloc[validation_end:].copy()

# Record the corresponding time boundaries for the report.
train_start = eta_model_df["pickup_timestamp"].iloc[0]
train_end_time = eta_model_df["pickup_timestamp"].iloc[train_end - 1]

val_start_time = eta_model_df["pickup_timestamp"].iloc[train_end]
val_end_time = eta_model_df["pickup_timestamp"].iloc[validation_end - 1]

test_start_time = eta_model_df["pickup_timestamp"].iloc[validation_end]
test_end_time = eta_model_df["pickup_timestamp"].iloc[-1]

print("Chronological split completed.\n")

print(
    f"Train:      {len(X_train):,} rows | "
    f"{train_start} to {train_end_time}"
)

print(
    f"Validation: {len(X_val):,} rows | "
    f"{val_start_time} to {val_end_time}"
)

print(
    f"Test:       {len(X_test):,} rows | "
    f"{test_start_time} to {test_end_time}"
)

Chronological split completed.

Train:      671,183 rows | 2025-04-01 00:00:31 to 2025-12-06 12:09:57
Validation: 143,825 rows | 2025-12-06 12:09:59 to 2026-02-01 15:04:59
Test:       143,825 rows | 2026-02-01 15:05:02 to 2026-03-31 23:59:49


## 5. Baseline and Tuned ETA Models

HistGradientBoostingRegressor is used because it handles nonlinear relationships efficiently on a large tabular dataset.

Two configurations are evaluated:

1. **Baseline model**: moderate complexity and conservative regularization.
2. **Tuned model**: increased tree complexity and iterations with a lower learning rate.

Model selection is based only on validation performance. The held-out test set is evaluated only after selecting the better configuration.

The primary metric is **MAE in minutes**, because it directly answers how many minutes the ETA prediction typically differs from the observed trip duration.

In [20]:
# Train two configurations using the same feature set and chronological split.
# The validation set is used for model selection; the test set remains untouched.

model_configs = {
    "HistGradientBoosting_Baseline": HistGradientBoostingRegressor(
        max_iter=300,
        learning_rate=0.05,
        max_leaf_nodes=31,
        l2_regularization=1.0,
        random_state=42
    ),
    "HistGradientBoosting_Tuned": HistGradientBoostingRegressor(
        max_iter=500,
        learning_rate=0.03,
        max_leaf_nodes=63,
        l2_regularization=2.0,
        random_state=42
    ),
}

eta_results = []
eta_models = {}

for model_name, model in model_configs.items():

    print(f"Training {model_name}...")

    model.fit(X_train, y_train)

    # Validation predictions are used for model selection.
    val_predictions = model.predict(X_val)

    val_mae = mean_absolute_error(y_val, val_predictions)
    val_rmse = np.sqrt(mean_squared_error(y_val, val_predictions))
    val_r2 = r2_score(y_val, val_predictions)

    eta_results.append({
        "model": model_name,
        "validation_mae_minutes": val_mae,
        "validation_rmse_minutes": val_rmse,
        "validation_r2": val_r2,
    })

    eta_models[model_name] = model

    print(
        f"  Validation MAE:  {val_mae:.4f} minutes\n"
        f"  Validation RMSE: {val_rmse:.4f} minutes\n"
        f"  Validation R²:   {val_r2:.4f}\n"
    )

eta_results_df = pd.DataFrame(eta_results)

print("Model comparison:")
display(eta_results_df)

Training HistGradientBoosting_Baseline...


  File "C:\Users\arudk\anaconda3\Lib\site-packages\joblib\externals\loky\backend\context.py", line 257, in _count_physical_cores
    cpu_info = subprocess.run(
               ^^^^^^^^^^^^^^^
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 548, in run
    with Popen(*popenargs, **kwargs) as process:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 1026, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "C:\Users\arudk\anaconda3\Lib\subprocess.py", line 1538, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


  Validation MAE:  5.1513 minutes
  Validation RMSE: 18.4677 minutes
  Validation R²:   0.3092

Training HistGradientBoosting_Tuned...
  Validation MAE:  5.1152 minutes
  Validation RMSE: 18.4879 minutes
  Validation R²:   0.3077

Model comparison:


,model,validation_mae_minutes,validation_rmse_minutes,validation_r2
0,HistGradientBoosting_Baseline,5.151344,18.467726,0.309230
1,HistGradientBoosting_Tuned,5.115159,18.487949,0.307716


## 6. Final ETA Evaluation on the Held-Out Test Set

The tuned model is selected using validation MAE.

Only after model selection do we evaluate it on the held-out test set. This provides an unbiased estimate of expected performance on future, unseen trips.

The primary result is reported in minutes so that the error has a direct operational interpretation.

In [23]:
# Select the model with the lowest validation MAE.
best_eta_row = eta_results_df.loc[
    eta_results_df["validation_mae_minutes"].idxmin()
]

best_eta_model_name = best_eta_row["model"]
best_eta_model = eta_models[best_eta_model_name]

print("Selected ETA model:", best_eta_model_name)

# Generate predictions only once the model has been selected.
eta_test_predictions = best_eta_model.predict(X_test)

# Final held-out test metrics.
eta_test_mae = mean_absolute_error(
    y_test,
    eta_test_predictions
)

eta_test_rmse = np.sqrt(
    mean_squared_error(
        y_test,
        eta_test_predictions
    )
)

eta_test_r2 = r2_score(
    y_test,
    eta_test_predictions
)

print("\nFinal held-out test performance:")
print(f"MAE:  {eta_test_mae:.4f} minutes")
print(f"RMSE: {eta_test_rmse:.4f} minutes")
print(f"R²:   {eta_test_r2:.4f}")

# A few predictions for a quick sanity check.
eta_prediction_check = pd.DataFrame({
    "actual_minutes": y_test.iloc[:15].values,
    "predicted_minutes": eta_test_predictions[:15]
})

eta_prediction_check["absolute_error_minutes"] = (
    eta_prediction_check["actual_minutes"]
    - eta_prediction_check["predicted_minutes"]
).abs()

print("\nSample predictions:")
display(eta_prediction_check)

Selected ETA model: HistGradientBoosting_Tuned

Final held-out test performance:
MAE:  4.5546 minutes
RMSE: 20.9420 minutes
R²:   0.2639

Sample predictions:


,actual_minutes,predicted_minutes,absolute_error_minutes
0,6.933333,11.521179,4.587846
1,11.900000,10.338758,1.561242
2,10.516667,14.601368,4.084701
3,6.000000,9.984580,3.984580
4,33.050000,23.089450,9.960550
5,3.900000,8.634352,4.734352
6,8.716667,9.674271,0.957604
7,25.300000,37.226042,11.926042
8,32.483333,22.666142,9.817191
9,7.000000,8.655997,1.655997


## 5. Enhanced ETA Feature Engineering

The initial model provides a useful baseline, but several variables can be represented more appropriately for travel-time prediction.

Zone identifiers represent categories rather than continuous measurements, while time-of-day is cyclical. Additional route-level features are therefore introduced to give the model a more informative representation of trip structure.

### Additional features

- Cyclical pickup-hour representation
- Cyclical day-of-week representation
- Same-origin-and-destination indicator
- Origin-destination route identifier
- Log-transformed trip distance

These features are constructed exclusively from information available at trip start.

No feature derived from `dropoff_timestamp`, actual duration, or calculated speed is introduced.

In [26]:
# Create enhanced features from information available before the trip starts.

enhanced_df = eta_model_df.copy()

# Time-of-day is cyclical: 23:00 and 00:00 are closer than their
# raw numerical values suggest.
enhanced_df["hour_sin"] = np.sin(
    2 * np.pi * enhanced_df["pickup_hour"] / 24
)

enhanced_df["hour_cos"] = np.cos(
    2 * np.pi * enhanced_df["pickup_hour"] / 24
)

# Day of week is also cyclical.
enhanced_df["dow_sin"] = np.sin(
    2 * np.pi * enhanced_df["pickup_day_of_week"] / 7
)

enhanced_df["dow_cos"] = np.cos(
    2 * np.pi * enhanced_df["pickup_day_of_week"] / 7
)

# Route structure.
enhanced_df["same_zone_trip"] = (
    enhanced_df["origin_loc_id"]
    == enhanced_df["dest_loc_id"]
).astype(int)

# A unique route identifier lets the model learn route-specific behaviour.
enhanced_df["route_id"] = (
    enhanced_df["origin_loc_id"].astype(int).astype(str)
    + "_"
    + enhanced_df["dest_loc_id"].astype(int).astype(str)
)

# Distance is highly right-skewed, so retain the original value while
# adding a compressed representation.
enhanced_df["log_distance_miles"] = np.log1p(
    enhanced_df["distance_miles"].clip(lower=0)
)

enhanced_features = [
    "provider_code",
    "rider_count",
    "distance_miles",
    "log_distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "same_zone_trip",
    "route_id",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_day_of_month",
    "pickup_month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]

print("Enhanced feature count:", len(enhanced_features))
print("\nNew features:")
for feature in [
    "log_distance_miles",
    "same_zone_trip",
    "route_id",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]:
    print(" -", feature)

print("\nRoute examples:")
display(
    enhanced_df[
        [
            "origin_loc_id",
            "dest_loc_id",
            "same_zone_trip",
            "route_id",
            "distance_miles",
            "log_distance_miles",
        ]
    ].head()
)

Enhanced feature count: 19

New features:
 - log_distance_miles
 - same_zone_trip
 - route_id
 - hour_sin
 - hour_cos
 - dow_sin
 - dow_cos

Route examples:


,origin_loc_id,dest_loc_id,same_zone_trip,route_id,distance_miles,log_distance_miles
0,138,164,0,138_164,8.20,2.219203
1,239,116,0,239_116,3.54,1.512927
2,234,148,0,234_148,2.25,1.178655
3,70,125,0,70_125,12.73,2.619583
4,254,94,0,254_94,4.84,1.764731


## 6. Prepare the Enhanced Modelling Matrix

The route identifier is categorical, so it is converted to a numeric category code for the tree-based model.

The resulting matrix contains only trip attributes and pickup-time information. Target-derived variables remain excluded.

In [29]:
# Convert the categorical route identifier into a compact integer code.
# The mapping is learned from the modelling sample and used consistently
# within this experiment.
enhanced_df["route_id_code"] = (
    enhanced_df["route_id"].astype("category").cat.codes
)

# Final numeric feature matrix.
enhanced_features = [
    "provider_code",
    "rider_count",
    "distance_miles",
    "log_distance_miles",
    "rate_class_id",
    "offline_record_flag",
    "origin_loc_id",
    "dest_loc_id",
    "same_zone_trip",
    "route_id_code",
    "pickup_hour",
    "pickup_day_of_week",
    "pickup_day_of_month",
    "pickup_month",
    "is_weekend",
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
]

X_enhanced = enhanced_df[enhanced_features].copy()
y_enhanced = enhanced_df["trip_duration_minutes"].copy()

# Keep the same chronological boundaries used for the original experiment.
X_enhanced = X_enhanced.loc[eta_model_df.index]
y_enhanced = y_enhanced.loc[eta_model_df.index]

X_train_enhanced = X_enhanced.iloc[:train_end].copy()
y_train_enhanced = y_enhanced.iloc[:train_end].copy()

X_val_enhanced = X_enhanced.iloc[train_end:validation_end].copy()
y_val_enhanced = y_enhanced.iloc[train_end:validation_end].copy()

X_test_enhanced = X_enhanced.iloc[validation_end:].copy()
y_test_enhanced = y_enhanced.iloc[validation_end:].copy()

print("Enhanced feature matrix:", X_enhanced.shape)
print("Training rows:", f"{len(X_train_enhanced):,}")
print("Validation rows:", f"{len(X_val_enhanced):,}")
print("Test rows:", f"{len(X_test_enhanced):,}")
print("Missing values:", X_enhanced.isna().sum().sum())

Enhanced feature matrix: (958833, 19)
Training rows: 671,183
Validation rows: 143,825
Test rows: 143,825
Missing values: 0


## 7. Enhanced Model Experiment

The enhanced feature set is evaluated against the original ETA feature set using the same chronological validation period.

The comparison is based on validation MAE, RMSE and R².

The original model acts as the benchmark. The enhanced model is retained only if it demonstrates an actual improvement rather than simply having more features.

This controlled comparison keeps the modelling decision reproducible and prevents unnecessary feature expansion.

In [34]:
# Train an enhanced model using the same general configuration as the
# selected tuned model from the first experiment.

enhanced_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.03,
    max_leaf_nodes=63,
    l2_regularization=2.0,
    random_state=42
)

print("Training enhanced ETA model...")

enhanced_model.fit(
    X_train_enhanced,
    y_train_enhanced
)

# Evaluate only on the validation period at this stage.
enhanced_val_predictions = enhanced_model.predict(
    X_val_enhanced
)

enhanced_val_mae = mean_absolute_error(
    y_val_enhanced,
    enhanced_val_predictions
)

enhanced_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_enhanced,
        enhanced_val_predictions
    )
)

enhanced_val_r2 = r2_score(
    y_val_enhanced,
    enhanced_val_predictions
)

print("\nEnhanced model validation performance:")
print(f"MAE:  {enhanced_val_mae:.4f} minutes")
print(f"RMSE: {enhanced_val_rmse:.4f} minutes")
print(f"R²:   {enhanced_val_r2:.4f}")

# Compare directly against the original tuned model.
original_val_mae = eta_results_df.loc[
    eta_results_df["model"] == "HistGradientBoosting_Tuned",
    "validation_mae_minutes"
].iloc[0]

original_val_rmse = eta_results_df.loc[
    eta_results_df["model"] == "HistGradientBoosting_Tuned",
    "validation_rmse_minutes"
].iloc[0]

original_val_r2 = eta_results_df.loc[
    eta_results_df["model"] == "HistGradientBoosting_Tuned",
    "validation_r2"
].iloc[0]

comparison_df = pd.DataFrame({
    "model": [
        "Original_Tuned",
        "Enhanced"
    ],
    "validation_mae_minutes": [
        original_val_mae,
        enhanced_val_mae
    ],
    "validation_rmse_minutes": [
        original_val_rmse,
        enhanced_val_rmse
    ],
    "validation_r2": [
        original_val_r2,
        enhanced_val_r2
    ]
})

print("\nValidation comparison:")
display(comparison_df)

mae_change = original_val_mae - enhanced_val_mae

print(
    f"\nValidation MAE improvement: {mae_change:.4f} minutes"
)

if mae_change > 0:
    print("Result: Enhanced features improve validation MAE.")
else:
    print("Result: Enhanced features do not improve validation MAE.")

Training enhanced ETA model...

Enhanced model validation performance:
MAE:  5.0980 minutes
RMSE: 18.4106 minutes
R²:   0.3135

Validation comparison:


,model,validation_mae_minutes,validation_rmse_minutes,validation_r2
0,Original_Tuned,5.115159,18.487949,0.307716
1,Enhanced,5.097998,18.410641,0.313493



Validation MAE improvement: 0.0172 minutes
Result: Enhanced features improve validation MAE.


## 8. Robust Target Transformation Experiment

Trip duration has a strongly right-skewed distribution. Most trips are relatively short, while a small number of observations have very long durations.

To reduce the influence of extreme observations during training, a second model is trained using:

`log1p(trip_duration_minutes)`

The predictions are transformed back using:

`expm1(prediction)`

Evaluation is still performed in the original minute scale, so MAE and RMSE remain directly interpretable.

This transformation affects only the training target. No information from the future or completed trip is introduced into the predictors.

In [37]:
# Train on a logarithmically transformed target.
# log1p keeps very short trips valid while reducing the influence of
# unusually long trips during optimization.

y_train_log = np.log1p(y_train_enhanced)

log_target_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.03,
    max_leaf_nodes=63,
    l2_regularization=2.0,
    loss="squared_error",
    random_state=42
)

print("Training log-target ETA model...")

log_target_model.fit(
    X_train_enhanced,
    y_train_log
)

# Convert predictions back to minutes.
log_val_predictions = np.expm1(
    log_target_model.predict(X_val_enhanced)
)

log_val_mae = mean_absolute_error(
    y_val_enhanced,
    log_val_predictions
)

log_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_enhanced,
        log_val_predictions
    )
)

log_val_r2 = r2_score(
    y_val_enhanced,
    log_val_predictions
)

print("\nLog-target model validation performance:")
print(f"MAE:  {log_val_mae:.4f} minutes")
print(f"RMSE: {log_val_rmse:.4f} minutes")
print(f"R²:   {log_val_r2:.4f}")

# Compare all ETA candidates tested so far.
eta_improvement_results = pd.DataFrame({
    "model": [
        "Original_Tuned",
        "Enhanced",
        "Enhanced_Log_Target"
    ],
    "validation_mae_minutes": [
        original_val_mae,
        enhanced_val_mae,
        log_val_mae
    ],
    "validation_rmse_minutes": [
        original_val_rmse,
        enhanced_val_rmse,
        log_val_rmse
    ],
    "validation_r2": [
        original_val_r2,
        enhanced_val_r2,
        log_val_r2
    ]
})

print("\nETA model comparison:")
display(eta_improvement_results)

Training log-target ETA model...

Log-target model validation performance:
MAE:  4.4570 minutes
RMSE: 18.2718 minutes
R²:   0.3238

ETA model comparison:


,model,validation_mae_minutes,validation_rmse_minutes,validation_r2
0,Original_Tuned,5.115159,18.487949,0.307716
1,Enhanced,5.097998,18.410641,0.313493
2,Enhanced_Log_Target,4.457002,18.271809,0.323808


## 9. MAE-Oriented ETA Model

The previous experiment showed that logarithmic target transformation substantially improves ETA prediction.

The final targeted experiment changes the training objective to absolute error so that the model is optimized more directly toward the primary evaluation metric, MAE.

This is compared against the current best log-target model using the same validation period.

The test set remains untouched until the final candidate is selected.

In [40]:
# Train an MAE-oriented model on the logarithmic target.
# Absolute-error optimization is less sensitive to extreme target values
# and aligns directly with the primary evaluation metric.

mae_target_model = HistGradientBoostingRegressor(
    max_iter=500,
    learning_rate=0.03,
    max_leaf_nodes=63,
    l2_regularization=2.0,
    loss="absolute_error",
    random_state=42
)

print("Training MAE-oriented log-target ETA model...")

mae_target_model.fit(
    X_train_enhanced,
    y_train_log
)

# Convert the model output back to minutes.
mae_val_predictions = np.expm1(
    mae_target_model.predict(X_val_enhanced)
)

mae_val_mae = mean_absolute_error(
    y_val_enhanced,
    mae_val_predictions
)

mae_val_rmse = np.sqrt(
    mean_squared_error(
        y_val_enhanced,
        mae_val_predictions
    )
)

mae_val_r2 = r2_score(
    y_val_enhanced,
    mae_val_predictions
)

print("\nMAE-oriented model validation performance:")
print(f"MAE:  {mae_val_mae:.4f} minutes")
print(f"RMSE: {mae_val_rmse:.4f} minutes")
print(f"R²:   {mae_val_r2:.4f}")

# Compare the strongest candidates.
final_eta_candidates = pd.DataFrame({
    "model": [
        "Original_Tuned",
        "Enhanced",
        "Enhanced_Log_Target",
        "Enhanced_Log_Target_MAE"
    ],
    "validation_mae_minutes": [
        original_val_mae,
        enhanced_val_mae,
        log_val_mae,
        mae_val_mae
    ],
    "validation_rmse_minutes": [
        original_val_rmse,
        enhanced_val_rmse,
        log_val_rmse,
        mae_val_rmse
    ],
    "validation_r2": [
        original_val_r2,
        enhanced_val_r2,
        log_val_r2,
        mae_val_r2
    ]
})

print("\nComplete ETA candidate comparison:")
display(
    final_eta_candidates.sort_values(
        "validation_mae_minutes"
    ).reset_index(drop=True)
)

Training MAE-oriented log-target ETA model...

MAE-oriented model validation performance:
MAE:  4.4577 minutes
RMSE: 18.3413 minutes
R²:   0.3187

Complete ETA candidate comparison:


,model,validation_mae_minutes,validation_rmse_minutes,validation_r2
0,Enhanced_Log_Target,4.457002,18.271809,0.323808
1,Enhanced_Log_Target_MAE,4.457745,18.341307,0.318655
2,Enhanced,5.097998,18.410641,0.313493
3,Original_Tuned,5.115159,18.487949,0.307716


In [42]:
# Final ETA evaluation
# The test set has remained untouched during model selection.
# We now evaluate the selected validation winner on unseen future data.

eta_winner = log_target_model

eta_test_predictions = np.expm1(
    eta_winner.predict(X_test_enhanced)
)

eta_test_mae = mean_absolute_error(
    y_test_enhanced,
    eta_test_predictions
)

eta_test_rmse = np.sqrt(
    mean_squared_error(
        y_test_enhanced,
        eta_test_predictions
    )
)

eta_test_r2 = r2_score(
    y_test_enhanced,
    eta_test_predictions
)

print("Final ETA model test performance:")
print(f"MAE:  {eta_test_mae:.4f} minutes")
print(f"RMSE: {eta_test_rmse:.4f} minutes")
print(f"R²:   {eta_test_r2:.4f}")

Final ETA model test performance:
MAE:  4.1480 minutes
RMSE: 20.8903 minutes
R²:   0.2676


In [44]:
# Save the selected ETA model and its metadata.
# The model was selected using validation performance only.
# The test set was used once for final evaluation.

eta_model_path = MODEL_DIR / "eta_model.pkl"
eta_metadata_path = MODEL_DIR / "eta_model_metadata.pkl"

joblib.dump(eta_winner, eta_model_path)

eta_metadata = {
    "model_name": "Enhanced_Log_Target",
    "target": "trip_duration_minutes",
    "target_transformation": "log1p",
    "prediction_inverse_transformation": "expm1",
    "features": list(X_train_enhanced.columns),
    "validation_mae_minutes": float(log_val_mae),
    "validation_rmse_minutes": float(log_val_rmse),
    "validation_r2": float(log_val_r2),
    "test_mae_minutes": float(eta_test_mae),
    "test_rmse_minutes": float(eta_test_rmse),
    "test_r2": float(eta_test_r2),
    "max_speed_mph_filter": 100,
    "random_state": RANDOM_STATE
}

joblib.dump(eta_metadata, eta_metadata_path)

print("ETA model saved successfully.")
print(f"Model:    {eta_model_path}")
print(f"Metadata: {eta_metadata_path}")
print()
print("Final test performance:")
print(f"MAE:  {eta_test_mae:.4f} minutes")
print(f"RMSE: {eta_test_rmse:.4f} minutes")
print(f"R²:   {eta_test_r2:.4f}")

ETA model saved successfully.
Model:    C:\Users\arudk\Downloads\UrbanFlow_AI\models\eta\eta_model.pkl
Metadata: C:\Users\arudk\Downloads\UrbanFlow_AI\models\eta\eta_model_metadata.pkl

Final test performance:
MAE:  4.1480 minutes
RMSE: 20.8903 minutes
R²:   0.2676


In [46]:
# Final ETA model summary for reporting and reproducibility.

eta_results = pd.DataFrame([{
    "model": "Enhanced_Log_Target",
    "target": "trip_duration_minutes",
    "validation_mae_minutes": log_val_mae,
    "validation_rmse_minutes": log_val_rmse,
    "validation_r2": log_val_r2,
    "test_mae_minutes": eta_test_mae,
    "test_rmse_minutes": eta_test_rmse,
    "test_r2": eta_test_r2,
    "training_rows": len(X_train_enhanced),
    "validation_rows": len(X_val_enhanced),
    "test_rows": len(X_test_enhanced)
}])

eta_results.to_csv(
    OUTPUT_DIR / "eta_model_results.csv",
    index=False
)

display(eta_results)

print("\nETA results saved:")
print(OUTPUT_DIR / "eta_model_results.csv")

,model,target,validation_mae_minutes,validation_rmse_minutes,validation_r2,test_mae_minutes,test_rmse_minutes,test_r2,training_rows,validation_rows,test_rows
0,Enhanced_Log_Target,trip_duration_minutes,4.457002,18.271809,0.323808,4.148047,20.890312,0.267568,671183,143825,143825



ETA results saved:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\eta_model_results.csv


In [48]:
# Consolidate the final model metrics for the project report.
# Each model has already been evaluated on its designated test set.

final_model_results = pd.DataFrame([
    {
        "task": "Base Fare Prediction",
        "model": "Tuned HistGradientBoosting",
        "validation_mae": 4.174575,
        "validation_rmse": 8.395087,
        "validation_r2": 0.788093,
        "test_mae": 3.931388,
        "test_rmse": 8.100816,
        "test_r2": 0.794363
    },
    {
        "task": "ETA Prediction",
        "model": "Enhanced Log-Target HistGradientBoosting",
        "validation_mae": log_val_mae,
        "validation_rmse": log_val_rmse,
        "validation_r2": log_val_r2,
        "test_mae": eta_test_mae,
        "test_rmse": eta_test_rmse,
        "test_r2": eta_test_r2
    },
    {
        "task": "Demand Forecasting",
        "model": "HistGradientBoosting",
        "validation_mae": 23.546835,
        "validation_rmse": 35.341348,
        "validation_r2": 0.933120,
        "test_mae": 22.514409,
        "test_rmse": 34.041943,
        "test_r2": 0.939418
    }
])

final_model_results.to_csv(
    OUTPUT_DIR / "final_model_results.csv",
    index=False
)

display(final_model_results)

print("\nSaved:")
print(OUTPUT_DIR / "final_model_results.csv")

,task,model,validation_mae,validation_rmse,validation_r2,test_mae,test_rmse,test_r2
0,Base Fare Prediction,Tuned HistGradientBoosting,4.174575,8.395087,0.788093,3.931388,8.100816,0.794363
1,ETA Prediction,Enhanced Log-Target HistGradientBoosting,4.457002,18.271809,0.323808,4.148047,20.890312,0.267568
2,Demand Forecasting,HistGradientBoosting,23.546835,35.341348,0.933120,22.514409,34.041943,0.939418



Saved:
C:\Users\arudk\Downloads\UrbanFlow_AI\outputs\final_model_results.csv
